# Zero-shot Whisper baseline on OpenSLR 104 (MUCS 2021 Hindi-English test)

This notebook measures how Whisper performs on Hinglish **before** fine-tuning.

**Steps**
1. Download the OpenSLR 104 Hindi-English **test** set (443 MB).
2. Cut the long recordings into utterances.
3. Run Whisper models with forced Hindi and with automatic language detection.
4. Normalize references and outputs with the same rules.
5. Calculate WER, CER, error types, script mix, and hallucinations.
6. Save every transcript and a summary table.

**Before you start:** Runtime → Change runtime type → **GPU** (T4 is enough).

No package installation is needed. Colab already has `torch`, `transformers`, `soundfile`, `scipy`, and `pandas`.

**License:** OpenSLR 104 is CC BY-SA 4.0. This is a test set: never train on it.

## 1. Settings

In [ ]:
from pathlib import Path

# Models to test (multilingual only, never ".en" checkpoints).
MODELS = [
    "openai/whisper-tiny",
    "openai/whisper-base",
    "openai/whisper-small",
    # "openai/whisper-medium",   # 769M parameters: slower, but useful as an upper reference
]

LANGUAGES = ["hi", "auto"]   # "hi" = forced Hindi, "auto" = automatic language detection
BEAMS = [1]                  # add 5 for beam search (about 2-3x slower): [1, 5]
BATCH_SIZE = 16              # lower it (8) if you get "CUDA out of memory"
MAX_NEW_TOKENS = 400         # Devanagari uses many tokens per word
MAX_UTTS = 200               # 0 = full test set. Start with 200 to check that everything works.
MIN_DUR, MAX_DUR = 0.5, 30.0 # seconds; Whisper reads 30 s at a time

DATA_DIR = None              # set to an extracted folder if you already have the data
DOWNLOAD_DIR = Path("data/openslr104")
CACHE_DIR = Path("data/openslr104_test_prepared")
OUT_DIR = Path("results/baseline_openslr104")

SAVE_TO_DRIVE = False        # True = copy results to Google Drive at the end

## 2. Check the GPU

In [ ]:
import torch, transformers, platform

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| python", platform.python_version())
if DEVICE == "cpu":
    print("WARNING: no GPU. Runtime -> Change runtime type -> GPU. On CPU, keep MAX_UTTS small.")

## 3. Download and extract the test set

In [ ]:
import tarfile, urllib.request

SAMPLE_RATE = 16000
TEST_ARCHIVE = "Hindi-English_test.tar.gz"
MIRRORS = [
    "https://www.openslr.org/resources/104/" + TEST_ARCHIVE,
    "https://us.openslr.org/resources/104/" + TEST_ARCHIVE,
]


def _progress(blocks, block_size, total):
    if total > 0:
        pct = min(100.0, blocks * block_size * 100.0 / total)
        print(f"\r{pct:5.1f}%", end="", flush=True)


def download_and_extract(download_dir: Path) -> Path:
    download_dir.mkdir(parents=True, exist_ok=True)
    archive = download_dir / TEST_ARCHIVE
    extract_dir = download_dir / "extracted"
    if extract_dir.exists() and any(extract_dir.iterdir()):
        print("Using existing folder:", extract_dir)
        return extract_dir
    if not archive.exists():
        last_err = None
        for url in MIRRORS:
            try:
                print("Downloading", url)
                tmp = archive.with_suffix(".part")
                urllib.request.urlretrieve(url, tmp, _progress)
                tmp.rename(archive)
                print()
                break
            except Exception as err:
                last_err = err
                print("\nFailed:", err)
        else:
            raise RuntimeError(f"All mirrors failed. Last error: {last_err}")
    print("Extracting", archive)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, "r:gz") as tar:
        try:
            tar.extractall(extract_dir, filter="data")   # safe extraction
        except TypeError:
            tar.extractall(extract_dir)
    return extract_dir


DATA_ROOT = Path(DATA_DIR) if DATA_DIR else download_and_extract(DOWNLOAD_DIR)
print("Data root:", DATA_ROOT)

## 4. Cut the recordings into utterances

The data is in Kaldi format: `text` (transcripts), `segments` (start and end times), and `wav.scp` (audio files).
This cell finds these files, cuts each utterance, and saves it as a 16 kHz WAV file with a `manifest.jsonl`.

In [ ]:
import json, re
from collections import defaultdict
import numpy as np
import soundfile as sf


def read_kaldi_map(path):
    out = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                parts = line.split(maxsplit=1)
                out[parts[0]] = parts[1] if len(parts) > 1 else ""
    return out


def read_segments(path):
    out = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            p = line.split()
            if len(p) >= 4:
                out[p[0]] = (p[1], float(p[2]), float(p[3]))
    return out


def find_kaldi_dir(root):
    cands = [p.parent for p in root.rglob("text") if p.is_file()]
    if not cands:
        raise FileNotFoundError(f"No Kaldi 'text' file under {root}")
    with_seg = [c for c in cands if (c / "segments").exists()]
    return (with_seg or cands)[0]


def resolve_wav(rec_id, scp_value, wav_index, base):
    if scp_value:
        for tok in [t for t in scp_value.split() if re.search(r"\.(wav|flac)$", t, re.I)]:
            p = Path(tok)
            for cand in (p, base / p):
                if cand.exists():
                    return cand
            if p.name in wav_index:
                return wav_index[p.name]
    for ext in (".wav", ".flac", ".WAV"):
        if rec_id + ext in wav_index:
            return wav_index[rec_id + ext]
    return None


def load_audio(path):
    audio, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != SAMPLE_RATE:
        from scipy.signal import resample_poly
        g = np.gcd(sr, SAMPLE_RATE)
        audio = resample_poly(audio, SAMPLE_RATE // g, sr // g).astype("float32")
    return audio


def prepare_utterances(data_root, cache_dir, min_dur, max_dur):
    manifest = cache_dir / "manifest.jsonl"
    if manifest.exists():
        print("Using existing manifest:", manifest)
        return manifest
    kdir = find_kaldi_dir(data_root)
    texts = read_kaldi_map(kdir / "text")
    segments = read_segments(kdir / "segments") if (kdir / "segments").exists() else {}
    scp = read_kaldi_map(kdir / "wav.scp") if (kdir / "wav.scp").exists() else {}
    wav_index = {}
    for p in data_root.rglob("*"):
        if p.suffix.lower() in (".wav", ".flac") and p.is_file():
            wav_index.setdefault(p.name, p)
    print(f"Kaldi folder: {kdir}")
    print(f"{len(texts)} transcripts | {len(segments)} segments | "
          f"{len(scp)} wav.scp entries | {len(wav_index)} audio files")

    by_rec = defaultdict(list)
    if segments:
        for utt, (rec, s, e) in segments.items():
            by_rec[rec].append((utt, s, e))
    else:
        for utt in texts:
            by_rec[utt].append((utt, None, None))

    wav_out = cache_dir / "wav"
    wav_out.mkdir(parents=True, exist_ok=True)
    stats, rows = defaultdict(int), []
    for rec, utts in sorted(by_rec.items()):
        path = resolve_wav(rec, scp.get(rec, ""), wav_index, kdir)
        if path is None:
            stats["missing_audio"] += len(utts)
            continue
        audio = load_audio(path)
        for utt, s, e in utts:
            text = texts.get(utt, "").strip()
            if not text:
                stats["no_text"] += 1
                continue
            clip = audio if s is None else audio[int(s * SAMPLE_RATE): int(e * SAMPLE_RATE)]
            dur = len(clip) / SAMPLE_RATE
            if dur < min_dur or dur > max_dur:
                stats["bad_duration"] += 1
                continue
            out_path = wav_out / f"{utt}.wav"
            sf.write(str(out_path), clip, SAMPLE_RATE, subtype="PCM_16")
            rows.append({"utt_id": utt, "rec_id": rec, "audio": str(out_path),
                         "duration": round(dur, 3), "text": text})
    with open(manifest, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    hours = sum(r["duration"] for r in rows) / 3600
    print(f"Kept {len(rows)} utterances ({hours:.2f} h). Skipped: no_text={stats['no_text']}, "
          f"bad_duration={stats['bad_duration']}, missing_audio={stats['missing_audio']}")
    if not rows:
        raise RuntimeError("No utterances prepared. Check the folder layout printed above.")
    return manifest


MANIFEST = prepare_utterances(DATA_ROOT, CACHE_DIR, MIN_DUR, MAX_DUR)

## 5. Text normalization

The same rules are applied to the references and to the model outputs.
Numbers are **not** converted between digits and words. Write this in your report.

In [ ]:
import unicodedata

_DEV_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")
_TAGS = re.compile(r"<[^>]*>|\[[^\]]*\]|\{[^}]*\}")

NORMALIZATION_RULES = """
1. Unicode NFC (also makes nukta letters consistent).
2. Remove tags such as <noise>, [music], {laugh}.
3. Devanagari digits -> ASCII digits.
4. Lowercase (affects Latin only).
5. Remove punctuation and symbols (Unicode P*, S*) and format characters (Cf, e.g. ZWJ).
   Devanagari vowel signs and virama are kept.
6. Collapse whitespace. Numbers are not converted between digits and words.
"""


def normalize(text):
    text = unicodedata.normalize("NFC", text)
    text = _TAGS.sub(" ", text)
    text = text.translate(_DEV_DIGITS).lower()
    chars = []
    for ch in text:
        cat = unicodedata.category(ch)
        if cat[0] in ("P", "S"):
            chars.append(" ")
        elif cat != "Cf":
            chars.append(ch)
    return re.sub(r"\s+", " ", "".join(chars)).strip()


def word_script(word):
    dev = lat = other = 0
    for ch in word:
        o = ord(ch)
        if 0x0900 <= o <= 0x097F or 0xA8E0 <= o <= 0xA8FF:
            dev += 1
        elif "a" <= ch <= "z":
            lat += 1
        elif ch.isdigit():
            continue
        elif ch.isalpha():
            other += 1          # for example Urdu/Arabic script
    if dev == lat == other == 0:
        return "digit"
    best = max(dev, lat, other)
    return "devanagari" if best == dev else "latin" if best == lat else "other"


# Quick check
for s in ["नमस्ते, आज हम bash script सीखेंगे।", "यह <noise> GChemPaint का tutorial है", "०१२ Version 4.1"]:
    n = normalize(s)
    print(f"{s!r:45} -> {n!r:40} {[word_script(w) for w in n.split()]}")

## 6. Metrics

WER and CER use word-level and character-level edit distance (the same definition as the `jiwer` library).
Error types: **substitutions** (wrong word), **deletions** (missing word), **insertions** (extra word).
A "possible hallucination" is an output with more than 2 × the reference words + 5.

In [ ]:
try:
    from rapidfuzz.distance import Levenshtein as _RFLev   # fast, if available
except ImportError:
    _RFLev = None


def align_counts(ref, hyp):
    """Return (substitutions, deletions, insertions, hits) for two token lists."""
    n, m = len(ref), len(hyp)
    # dp[i][j] = (cost, S, D, I)
    prev = [(j, 0, 0, j) for j in range(m + 1)]
    for i in range(1, n + 1):
        cur = [(i, 0, i, 0)] + [None] * m
        for j in range(1, m + 1):
            if ref[i - 1] == hyp[j - 1]:
                cur[j] = prev[j - 1]
            else:
                s, d, ins = prev[j - 1], prev[j], cur[j - 1]
                best = min((s[0], 0), (d[0], 1), (ins[0], 2))
                if best[1] == 0:
                    cur[j] = (s[0] + 1, s[1] + 1, s[2], s[3])
                elif best[1] == 1:
                    cur[j] = (d[0] + 1, d[1], d[2] + 1, d[3])
                else:
                    cur[j] = (ins[0] + 1, ins[1], ins[2], ins[3] + 1)
        prev = cur
    cost, S, D, I = prev[m]
    return S, D, I, n - S - D


def char_distance(a, b):
    if _RFLev is not None:
        return _RFLev.distance(a, b)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def compute_metrics(refs, hyps):
    S = D = I = N = 0
    char_err = char_total = 0
    halluc = empty = 0
    ref_sc, hyp_sc = defaultdict(int), defaultdict(int)
    per_utt_wer = []
    for r, h in zip(refs, hyps):
        rw, hw = r.split(), h.split()
        s, d, i, _ = align_counts(rw, hw)
        S, D, I, N = S + s, D + d, I + i, N + len(rw)
        per_utt_wer.append(100 * (s + d + i) / max(1, len(rw)))
        char_err += char_distance(r, h)
        char_total += len(r)
        halluc += len(hw) > 2 * len(rw) + 5
        empty += not h
        for w in rw:
            ref_sc[word_script(w)] += 1
        for w in hw:
            hyp_sc[word_script(w)] += 1
    rt, ht = max(1, sum(ref_sc.values())), max(1, sum(hyp_sc.values()))
    return {
        "wer": 100 * (S + D + I) / N,
        "cer": 100 * char_err / char_total,
        "sub_rate": 100 * S / N, "del_rate": 100 * D / N, "ins_rate": 100 * I / N,
        "ref_latin_pct": 100 * ref_sc["latin"] / rt,
        "hyp_latin_pct": 100 * hyp_sc["latin"] / ht,
        "hyp_devanagari_pct": 100 * hyp_sc["devanagari"] / ht,
        "hyp_other_script_pct": 100 * hyp_sc["other"] / ht,
        "possible_hallucinations": int(halluc), "empty_outputs": int(empty),
    }, per_utt_wer


# Self-test: 1 substitution + 1 deletion + 1 insertion over 4 reference words = 75% WER
_m, _ = compute_metrics(["a b c d"], ["a x c e f"])
assert round(_m["wer"], 2) == 75.0, _m
print("Metric self-test passed:", {k: round(v, 2) for k, v in _m.items() if k.endswith("rate") or k == "wer"})

## 7. Load the utterances

In [ ]:
def read_manifest(path, max_utts=0):
    with open(path, encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
    if max_utts and max_utts < len(rows):
        rng = np.random.default_rng(0)        # fixed subset: the same for every model
        idx = sorted(rng.choice(len(rows), size=max_utts, replace=False))
        rows = [rows[i] for i in idx]
    return rows


ROWS = read_manifest(MANIFEST, MAX_UTTS)
REFS_NORM = [normalize(r["text"]) for r in ROWS]
_keep = [i for i, t in enumerate(REFS_NORM) if t]
ROWS = [ROWS[i] for i in _keep]
REFS_NORM = [REFS_NORM[i] for i in _keep]
HOURS = sum(r["duration"] for r in ROWS) / 3600
print(f"{len(ROWS)} utterances, {HOURS:.2f} h")
for r in ROWS[:3]:
    print(r["utt_id"], r["duration"], "s |", r["text"])

## 8. Whisper inference

In [ ]:
import time
from transformers import WhisperForConditionalGeneration, WhisperProcessor


class WhisperRunner:
    def __init__(self, model_id, device):
        if model_id.endswith(".en"):
            raise ValueError(f"{model_id} is English-only. Use a multilingual checkpoint.")
        self.device = device
        self.dtype = torch.float16 if device == "cuda" else torch.float32
        self.processor = WhisperProcessor.from_pretrained(model_id)
        self.model = WhisperForConditionalGeneration.from_pretrained(model_id)
        self.model = self.model.to(device, dtype=self.dtype).eval()

    def transcribe(self, audios, language, num_beams, max_new_tokens):
        feats = self.processor.feature_extractor(
            audios, sampling_rate=SAMPLE_RATE, return_tensors="pt").input_features
        feats = feats.to(self.device, dtype=self.dtype)
        with torch.inference_mode():
            ids = self.model.generate(
                feats,
                language=language,      # "hi" or None (auto-detect)
                task="transcribe",
                num_beams=num_beams,
                do_sample=False,        # temperature 0
                max_new_tokens=max_new_tokens,
            )
        texts = self.processor.batch_decode(ids, skip_special_tokens=True)
        raw = self.processor.batch_decode(ids, skip_special_tokens=False)
        langs = []
        for r in raw:
            m = re.search(r"<\|([a-z]{2,3}(?:-[a-z]+)?)\|>", r)
            langs.append(m.group(1) if m else None)
        return [t.strip() for t in texts], langs


def run_setting(runner, rows, language, num_beams):
    hyps, langs = [], []
    audio_sec = sum(r["duration"] for r in rows)
    t0 = time.perf_counter()
    for i in range(0, len(rows), BATCH_SIZE):
        batch = rows[i:i + BATCH_SIZE]
        texts, detected = runner.transcribe([load_audio(r["audio"]) for r in batch],
                                            language, num_beams, MAX_NEW_TOKENS)
        hyps += texts
        langs += detected
        print(f"\r    {min(i + BATCH_SIZE, len(rows))}/{len(rows)}", end="", flush=True)
    print()
    return hyps, langs, (time.perf_counter() - t0) / max(audio_sec, 1e-9)

## 9. Run the baseline

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "run_info.json").write_text(json.dumps({
    "dataset": "OpenSLR 104 / MUCS 2021 Hindi-English test",
    "n_utterances": len(ROWS), "hours": round(HOURS, 3),
    "settings": {"models": MODELS, "languages": LANGUAGES, "beams": BEAMS,
                 "batch_size": BATCH_SIZE, "max_new_tokens": MAX_NEW_TOKENS,
                 "max_utts": MAX_UTTS, "min_dur": MIN_DUR, "max_dur": MAX_DUR},
    "versions": {"python": platform.python_version(), "torch": torch.__version__,
                 "transformers": transformers.__version__},
    "device": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu",
    "normalization": NORMALIZATION_RULES.strip(),
}, indent=2, ensure_ascii=False))

SUMMARY = []
for model_id in MODELS:
    print(f"\n=== {model_id} ===")
    runner = WhisperRunner(model_id, DEVICE)
    for lang in LANGUAGES:
        for beams in BEAMS:
            tag = f"{model_id.split('/')[-1]}__lang-{lang}__beam-{beams}"
            print(f"  {tag}")
            hyps, detected, rtf = run_setting(runner, ROWS, None if lang == "auto" else lang, beams)
            hyps_norm = [normalize(h) for h in hyps]
            m, per_utt = compute_metrics(REFS_NORM, hyps_norm)
            counts = defaultdict(int)
            for d in detected:
                counts[d or "unknown"] += 1
            top = ", ".join(f"{k}:{v}" for k, v in sorted(counts.items(), key=lambda kv: -kv[1])[:4])
            SUMMARY.append({"model": model_id, "language": lang, "beams": beams, **m,
                            "rtf_gpu": rtf, "detected_langs": top, "tag": tag})
            print(f"    WER {m['wer']:.2f} | CER {m['cer']:.2f} | Latin words ref/hyp "
                  f"{m['ref_latin_pct']:.1f}/{m['hyp_latin_pct']:.1f}% | other script "
                  f"{m['hyp_other_script_pct']:.1f}% | RTF {rtf:.3f}")
            with open(OUT_DIR / f"{tag}.jsonl", "w", encoding="utf-8") as f:
                for r, h, rn, hn, d, w in zip(ROWS, hyps, REFS_NORM, hyps_norm, detected, per_utt):
                    f.write(json.dumps({"utt_id": r["utt_id"], "duration": r["duration"],
                                        "ref": r["text"], "hyp": h, "ref_norm": rn, "hyp_norm": hn,
                                        "detected_lang": d, "utt_wer": round(w, 2)},
                                       ensure_ascii=False) + "\n")
    del runner
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
print("\nDone.")

## 10. Results table

- **WER / CER / Sub / Del / Ins** are percentages of the reference words or characters.
- **Latin % (hyp)**: compare with the reference value printed below the table. If it is much lower, Whisper writes English words in Devanagari.
- **Other script %**: words in a third script (for example Urdu). This is a wrong-script error.
- **RTF (GPU)**: processing time ÷ audio time on the Colab GPU. It is **not** the speed on a phone.

In [ ]:
import pandas as pd

df = pd.DataFrame(SUMMARY)
cols = ["model", "language", "beams", "wer", "cer", "sub_rate", "del_rate", "ins_rate",
        "hyp_latin_pct", "hyp_devanagari_pct", "hyp_other_script_pct",
        "possible_hallucinations", "empty_outputs", "rtf_gpu", "detected_langs"]
df[cols].to_csv(OUT_DIR / "summary.csv", index=False)
try:
    table_text = df[cols].round(2).to_markdown(index=False)   # needs the "tabulate" package
except ImportError:
    table_text = df[cols].round(2).to_string(index=False)
(OUT_DIR / "summary.md").write_text(table_text, encoding="utf-8")
print(f"Reference: {SUMMARY[0]['ref_latin_pct']:.1f}% of words are in Latin script.")
display(df[cols].round(2))

## 11. Look at the worst examples

In [ ]:
SHOW_TAG = SUMMARY[0]["tag"]   # change to any tag from the table
N_SHOW = 10

with open(OUT_DIR / f"{SHOW_TAG}.jsonl", encoding="utf-8") as f:
    utts = [json.loads(line) for line in f]
print("Run:", SHOW_TAG)
for u in sorted(utts, key=lambda u: -u["utt_wer"])[:N_SHOW]:
    print(f"\n{u['utt_id']} | {u['duration']} s | WER {u['utt_wer']} | lang {u['detected_lang']}")
    print("  REF:", u["ref_norm"])
    print("  HYP:", u["hyp_norm"])

## 12. (Optional) Save results to Google Drive

In [ ]:
if SAVE_TO_DRIVE:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive") / OUT_DIR.name
    shutil.copytree(OUT_DIR, dest, dirs_exist_ok=True)
    print("Copied to", dest)
else:
    print("SAVE_TO_DRIVE is False. Results are in", OUT_DIR.resolve(),
          "(Colab deletes them when the session ends).")

## Notes for your report

- **Fixed subset:** with `MAX_UTTS > 0`, every model uses the same random utterances (seed 0).
- **Removed clips:** clips longer than 30 s or without text are removed. Report the number from section 4.
- **Label errors:** some MUCS test transcripts are misaligned with the audio. Check the worst examples by listening before you blame the model.
- **No training on this set:** remove any training clip whose normalized transcript matches one of these references.